# 🎬 AI Cloud Documentary Engine v3.0 (Dhruv Rathee Style + Expressive Indic AI)
**Up to 25 Min | Local Gemma-3 | IndicF5 4-Mood Voice | MMS-300M CTC Span-ID Sync | Playwright Cards | FLUX + Hero LTX I2V | Drive Auto-Resume**

Is notebook se aap Google Colab GPU (L4 / T4) par bilkul **real human storytelling**, **sahi Hindi matra**, **45:25:25:5 hybrid visuals**, aur **crash-proof Google Drive checkpointing** ke saath 25 minute tak ki 1080p/4K documentary bana sakte hain.

In [ ]:
#@title 🛠️ Step 1: System Setup, Static Hindi Fonts, Playwright & Colab GPU Dependencies
import os
import sys

print('📦 Installing FFmpeg, Pango, HarfBuzz, libass9 & Devanagari shaping libs...')
os.system('apt-get update -qq && apt-get install -y -qq ffmpeg fontconfig fonts-dejavu-core fonts-noto-core fonts-lohit-deva libpango-1.0-0 libpangocairo-1.0-0 libharfbuzz0b libharfbuzz-dev libfribidi0 libfribidi-dev libraqm-dev libass9')

repo_dir = '/content/ai-cloud-video-engine'
if not os.path.exists(repo_dir):
    print('📥 Cloning repository from GitHub...')
    os.system(f'git clone https://github.com/ankitputra160-jpg/ai-cloud-video-engine.git {repo_dir}')
else:
    print('🔄 Updating repository to latest commit...')
    os.system(f'cd {repo_dir} && git pull origin main')

os.chdir(repo_dir)
sys.path.insert(0, repo_dir)
os.system('mkdir -p ~/.local/share/fonts && cp -f fonts/*.ttf ~/.local/share/fonts/ && fc-cache -fv > /dev/null')

print('📦 Installing Core + Colab GPU Python packages (IndicF5, CTC Aligner, FLUX, Playwright)...')
os.system(f'pip install -q -r {repo_dir}/requirements.txt')
os.system(f'pip install -q -r {repo_dir}/requirements_colab.txt')
os.system('playwright install chromium && playwright install-deps chromium 2>/dev/null || true')
print('✅ Colab v3.0 GPU Environment Ready!')

In [ ]:
#@title ☁️ Step 2: Mount Google Drive + Bootstrap 4-Mood CC0 BGM Bank & Voice Folder
from google.colab import drive
import os, sys

drive.mount('/content/drive')
base_drive = '/content/drive/MyDrive/AI_Documentaries'
bgm_bank = os.path.join(base_drive, 'bgm_bank')
voice_ref = os.path.join(base_drive, 'voice_ref')
workspaces_root = os.path.join(base_drive, 'workspaces')
for d in (base_drive, bgm_bank, voice_ref, workspaces_root):
    os.makedirs(d, exist_ok=True)

repo_dir = '/content/ai-cloud-video-engine'
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

from engine.audio.bgm import bootstrap_cc0_bgm_bank
print('🎵 Checking / downloading CC0 documentary BGM tracks to Drive cache...')
bank = bootstrap_cc0_bgm_bank(bgm_bank)
print('✅ Google Drive Connected!')
print(f'   📁 Outputs & Checkpoints: {base_drive}')
print(f'   🎙️ Reference Voice Bank (replace ref_*.wav anytime): {voice_ref}')
print(f'   🎼 CC0 BGM Mood Folders Ready: { {k: len(v) for k, v in bank.items()} }')

In [ ]:
#@title 🎥 Step 3: Enter Topic, Optional Keys & Generate / Resume Documentary { run: "auto" }
Topic = "CJI Controversy & CJP Protest Full Timeline" #@param {type:"string"}
Language = "hi" #@param ["hi", "en"]
Duration = 180 #@param {type:"slider", min:10, max:1500, step:10}
Resolution = "1080p" #@param ["1080p", "720p", "4k"]
Style = "Documentary" #@param ["Documentary", "2D Animation", "Vox Explainer", "Historical", "Scientific", "Biographical"]
Subtitle_Style = "ass_karaoke" #@param ["ass_karaoke", "pill_box", "srt_classic"]
LLM_Mode = "auto" #@param ["auto", "local_gemma3", "gemini", "groq", "deterministic"]
Voice_Engine = "auto" #@param ["auto", "indicf5", "fish", "sarvam", "cartesia", "indic_parler", "qwen3_tts", "elevenlabs", "edge", "piper"]
Enable_AI_Video_Hooks = True #@param {type:"boolean"}
Preview_Hook_Only = False #@param {type:"boolean"}
Resume_Job_Slug = "" #@param {type:"string"}

#@markdown ### 🎙️ Custom Voice Cloning (Optional — ya Drive ke `voice_ref/` me `.mp3`/`.wav` + `.txt` rakhein)
Custom_Voice_File = "" #@param {type:"string"}
Custom_Voice_Text = "" #@param {type:"string"}

#@markdown ### 🔑 Optional Tokens / API Keys (ya left sidebar 🔑 Secrets me save karein)
HF_TOKEN = "" #@param {type:"string"}
GEMINI_API_KEY = "" #@param {type:"string"}
GROQ_API_KEY = "" #@param {type:"string"}
FISH_API_KEY = "" #@param {type:"string"}
FISH_REFERENCE_ID = "" #@param {type:"string"}
SARVAM_API_KEY = "" #@param {type:"string"}
CARTESIA_API_KEY = "" #@param {type:"string"}
CARTESIA_VOICE_ID = "" #@param {type:"string"}
ELEVENLABS_API_KEY = "" #@param {type:"string"}
ELEVENLABS_VOICE_ID = "" #@param {type:"string"}

import os, re, sys, time, subprocess
from IPython.display import Image, Video, display

repo_dir = '/content/ai-cloud-video-engine'
if not os.path.exists(repo_dir):
    os.system(f'git clone https://github.com/ankitputra160-jpg/ai-cloud-video-engine.git {repo_dir}')
else:
    os.system(f'cd {repo_dir} && git pull origin main -q')
os.chdir(repo_dir)
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

def _resolve_secret(form_val, secret_name):
    if form_val and form_val.strip():
        return form_val.strip()
    try:
        from google.colab import userdata
        val = userdata.get(secret_name)
        if val:
            return str(val).strip()
    except Exception:
        pass
    return os.environ.get(secret_name, '').strip()

env = os.environ.copy()
env['PYTHONPATH'] = repo_dir
for k, form_v in (
    ('HF_TOKEN', HF_TOKEN),
    ('GEMINI_API_KEY', GEMINI_API_KEY),
    ('GROQ_API_KEY', GROQ_API_KEY),
    ('FISH_API_KEY', FISH_API_KEY),
    ('FISH_REFERENCE_ID', FISH_REFERENCE_ID),
    ('SARVAM_API_KEY', SARVAM_API_KEY),
    ('CARTESIA_API_KEY', CARTESIA_API_KEY),
    ('CARTESIA_VOICE_ID', CARTESIA_VOICE_ID),
    ('ELEVENLABS_API_KEY', ELEVENLABS_API_KEY),
    ('ELEVENLABS_VOICE_ID', ELEVENLABS_VOICE_ID),
):
    resolved = _resolve_secret(form_v, k)
    if resolved:
        env[k] = resolved

safe_name = re.sub(r'[^a-zA-Z0-9_]', '_', Topic)[:32].strip('_') or 'documentary'
job_slug = Resume_Job_Slug.strip() or f'{safe_name}_{Language}_{Duration}s'
base_drive = '/content/drive/MyDrive/AI_Documentaries' if os.path.exists('/content/drive/MyDrive') else '/content/output'
workspace_dir = os.path.join(base_drive, 'workspaces', job_slug)
voice_ref_dir = os.path.join(base_drive, 'voice_ref')
bgm_bank_dir = os.path.join(base_drive, 'bgm_bank')
for d in (base_drive, workspace_dir, voice_ref_dir, bgm_bank_dir):
    os.makedirs(d, exist_ok=True)

final_output_path = os.path.join(base_drive, f'{job_slug}.mp4')
thumb_output_path = os.path.join(base_drive, f'{job_slug}_thumbnail.jpg')

print('=' * 64)
print(f'🚀 Starting v3.0 Production: {Topic} ({Style})')
print(f'🧠 LLM: {LLM_Mode} | 🎙️ Voice: {Voice_Engine} | ⏱️ Target: {Duration}s | Preview: {Preview_Hook_Only}')
print(f'💾 Drive Checkpoint Workspace (Auto-Resume): {workspace_dir}')
print(f'📁 Final Video: {final_output_path}')
print('=' * 64)

cmd = [
    sys.executable, '-m', 'engine.pipeline',
    '--topic', str(Topic),
    '--language', str(Language),
    '--duration', str(Duration),
    '--resolution', str(Resolution),
    '--style', str(Style),
    '--subtitle-style', str(Subtitle_Style),
    '--llm-mode', str(LLM_Mode),
    '--voice-engine', str(Voice_Engine),
    '--voice-bank-dir', voice_ref_dir,
    '--bgm-bank-dir', bgm_bank_dir,
    '--thumbnail', thumb_output_path,
    '--output', final_output_path,
    '--work-dir', workspace_dir,
]
if Custom_Voice_File and Custom_Voice_File.strip():
    cmd.extend(['--voice-ref', Custom_Voice_File.strip()])
if Custom_Voice_Text and Custom_Voice_Text.strip():
    cmd.extend(['--voice-ref-text', Custom_Voice_Text.strip()])
if Preview_Hook_Only:
    cmd.append('--preview-only')
if not Enable_AI_Video_Hooks:
    cmd.append('--no-ai-video')

process = subprocess.Popen(cmd, cwd=repo_dir, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode == 0 and os.path.exists(final_output_path):
    size_mb = os.path.getsize(final_output_path) / (1024 * 1024)
    print('\n' + '=' * 64)
    print(f'🏆 VIDEO READY! Size: {size_mb:.2f} MB | Saved in Google Drive!')
    print('=' * 64)
    if os.path.exists(thumb_output_path):
        display(Image(filename=thumb_output_path, width=480))
    display(Video(final_output_path, embed=True, width=720))
else:
    print(f'\n❌ Video generation stopped (code: {process.returncode}). Re-run Step 3 with the same Topic/Resume_Job_Slug to continue from Drive checkpoint!')